# Simulation Code T = 2

## Import libraries

In [1]:
from utils.dynamicRieszFunctions import estimateDynamicRiesz_all
from utils.dynamicRieszFunctions import estimateDynamicRiesz
from utils.dynamicRieszBradic import estimateBradic

import torch
import pandas as pd
import time
from torch.distributions import Normal

/Users/wisserutgers/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Estimation Settings

In [2]:
lasso_a_settings = {
    'estimate' : True,
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'estimate' : True,
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' :  "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'estimate' : True,
    'poly_degree' : 0,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}
rf_f_settings = {
    'estimate' : True,
    'poly_degree' : 1,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'estimate' : True,
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'estimate' : True,
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}


## Generate data 

##### Simulation settings:

In [3]:
torch.manual_seed(123);

# Parameters
N = 1000
tmax = 20

# Higher dimensions = more sparsity. Minimum is dimX1 = dimX2 = 1
dimX1 = 2
dimX2 = 2

##### Define treatment probability function

In [4]:
# Bounds (only for truncated distributions)
lower = 0.05
upper = 0.95

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

# Simple nonlinear probability (from adversarial Riesz paper)
def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)

# Simple nonlinear probability (from adversarial Riesz paper)
def double_nonlinear(x):
    return lower + (upper - lower) * ((x < - 1/2) + (x < 1/2))

In [5]:
treatment_probability_func = truncated_adv

##### Coefficients

In [6]:
# Treatment probability period 1
beta_pi1_0 = 0 # Intercept 
beta_pi1_X1 = torch.tensor([1] + [0] * (dimX1 - 1), dtype=torch.float32).reshape(-1,1) # Effect of X1 

# Treatment probability period 2
beta_pi2_0_0 = 0 # Intercept if untreated in period 1
beta_pi2_0_X2 = torch.tensor([0.5] + [0] * (dimX2 - 1), dtype=torch.float32).reshape(-1,1) # Effect of X2 if untreated in period 1
beta_pi2_1_0 = 0 # Intercept if treated in period 1
beta_pi2_1_X2 = torch.tensor([1] + [0] * (dimX2 - 1), dtype=torch.float32).reshape(-1,1) # Effect of X2 if treated in period 1

# Outcome
c1 = 2.2
c2 = 1.2
c12 = 0.5

beta1 = 1.2
beta2 = 0.8

delta1 = 1
delta2 = 1
delta12 = 0.5

##### Simulate data

In [ ]:
# Period 1
X1_all = torch.randn(N * tmax, dimX1) # Normally distributed X1 covariates
pi1_all = treatment_probability_func(beta_pi1_0 + X1_all @ beta_pi1_X1).reshape(-1,1) # Generated treatment probablities
D1_all = torch.bernoulli(pi1_all).int().reshape(-1,1) # Generate treatment assignment period 1

# Period 2
delta1_all = torch.randn(N * tmax,1) # Change from X1 to X2 if treated in period 1
delta2_all = torch.randn(N * tmax, dimX2) # Change from X1 to X2 always

X2_all = X1_all + D1_all * (1 + delta1_all) + delta2_all # Period 2 covariates
X2_1_all = X1_all + 1 + delta1_all + delta2_all # Period 2 covariates if treated in period 1
X2_0_all = X1_all + delta2_all # Period 2 covariates if untreated in period 1

pi2_0_all = treatment_probability_func(beta_pi2_0_0 + X2_0_all @ beta_pi2_0_X2) # Treatment probability if untreated in period 1
pi2_1_all = treatment_probability_func(beta_pi2_1_0 + X2_1_all @ beta_pi2_1_X2) # Treatment probability if treated in period 1
pi2_all = (1 - D1_all) * pi2_0_all + D1_all * pi2_1_all # Actual treatment probability

D2_all = torch.bernoulli(pi2_all).int() # Generate treatment assignment period 1

# Outcome
# g_00_all = 1 + X1_all[:,:1]
# g_11_all = (c1 * D1_all + c2 * D2_all + c12 * D1_all * D2_all + 
#         beta1 * (X1_all[:,:1] > 0) + beta2 * (X2_1_all[:,:1] > 0) + 
#         delta1 * D1_all * X1_all[:,:1] + delta2 * D2_all * X2_1_all[:,:1] + delta12 * D1_all * D2_all * X2_1_all[:,:1])

g_all = (X1_all[:,:1] + c1 * D1_all + c2 * D2_all + c12 * D1_all * D2_all + 
# g_all = (1 + X1_all[:,:1] + c1 * D1_all + c2 * D2_all + c12 * D1_all * D2_all + 
        beta1 * (X1_all[:,:1] > 0) + beta2 * (X2_all[:,:1] > 0) + 
        delta1 * D1_all * X1_all[:,:1] + delta2 * D2_all * X2_all[:,:1] + delta12 * D1_all * D2_all * X2_all[:,:1])
zeta_all = torch.randn(N * tmax,1) # Noise to outcome
Y_all = g_all + zeta_all


#### True values:

In [8]:
normal_dist1 = Normal(1, torch.sqrt(torch.tensor(3)))
cdf_value1 = 1 - normal_dist1.cdf(torch.tensor(0))


normal_dist0 = Normal(0, torch.sqrt(torch.tensor(2)))
cdf_value0 = 1 -  normal_dist0.cdf(-1 - X1_all[:,:1])


normal_dist0 = Normal(0, torch.sqrt(torch.tensor(2)))
cdf_value01 = 1 -  normal_dist0.cdf(- X1_all[:,:1])

mu2_1_all = ( X1_all[:,:1] + c1 * 1 + c2 * 1 + c12 * 1 * 1 + 
        beta1 * (X1_all[:,:1] > 0) + beta2 * (X2_1_all[:,:1] > 0) + 
        delta1 * 1 * X1_all[:,:1] + delta2 * 1 * X2_1_all[:,:1] + delta12 * 1 * 1 * X2_1_all[:,:1])

mu2_0_all = ( X1_all[:,:1] + c1 * 0 + c2 * 0 + c12 * 0 * 0 + 
        beta1 * (X1_all[:,:1] > 0) + beta2 * (X2_0_all[:,:1] > 0) + 
        delta1 * 0 * X1_all[:,:1] + delta2 * 0 * X2_0_all[:,:1] + delta12 * 0 * 0  * X2_0_all[:,:1])

mu1_1_all = ( X1_all[:,:1] + c1 * 1 + c2 * 1 + c12 * 1 * 1 + 
        beta1 * (X1_all[:,:1] > 0) + beta2 * cdf_value0 + 
        delta1 * 1 * X1_all[:,:1] + delta2 * 1 * (1 + X1_all[:,:1]) + delta12 * 1 * 1  * (1 + X1_all[:,:1]))

mu1_0_all = (  X1_all[:,:1] + c1 * 0 + c2 * 0 + c12 * 0 * 0 + 
        beta1 * (X1_all[:,:1] > 0) + beta2 * (X1_all[:,:1] > 0) + 
        delta1 * 0 * X1_all[:,:1] + delta2 * 0 * X1_all[:,:1] + delta12 * 0 * 0 * X1_all[:,:1])

print(mu2_1_all.mean(), mu2_0_all.mean(), mu1_1_all.mean(), mu1_0_all.mean())

theta0 =  0.5 * beta1 + 0.5 * beta2
theta1 = (c1 + c2 + c12 + 
        0.5 * beta1 + cdf_value1 * beta2 +
        (delta2 + delta12) * 1)

print(theta0, theta1)


tensor(6.5520) tensor(1.0016) tensor(6.5709) tensor(1.0031)
1.0 tensor(6.5745)


## Estimation:

#### Estimation settings

In [9]:
folds = 4

time0 = time.time()

#### Estimation 

In [10]:
pred_theta = torch.zeros(tmax,11)
pred_sig = torch.zeros(tmax,11)

RR1 = torch.zeros(N,tmax,11)
RR2 = torch.zeros(N,tmax,11)
f1 = torch.zeros(N,tmax,11)
f2 = torch.zeros(N,tmax,11)

for t in range(0,tmax):

    # Get data for current iteration
    X1, X2 = X1_all[(t) * N : (t+1) * N, :], X2_all[(t) * N : (t+1) * N, :]
    D1, D2 = D1_all[(t) * N : (t+1) * N], D2_all[(t) * N : (t+1) * N]
    Y = Y_all[(t) * N : (t+1) * N]

    X = torch.hstack((X1,X2))
    X_index = torch.tensor([X1.shape[1]-1,X1.shape[1]+X2.shape[1]-1])
    D = torch.hstack((D1,D2))

    # Set counterfactual of interest
    d = 1 * torch.ones(D.shape) 

    #---------------------------------------------------------------------------
    # Estimator 1: Oracle
    pi1, pi2_0, pi2_1  = pi1_all[(t) * N : (t+1) * N], pi2_0_all[(t) * N : (t+1) * N], pi2_1_all[(t) * N : (t+1) * N]
    mu2_1, mu2_0, mu1_1, mu1_0 = mu2_1_all[(t) * N : (t+1) * N], mu2_0_all[(t) * N : (t+1) * N], mu1_1_all[(t) * N : (t+1) * N], mu1_0_all[(t) * N : (t+1) * N]
    if d[0,0] == 1:
        RR2[:,t,:1], RR1[:,t,:1] = D1 * D2 / (pi1 * pi2_1), D1 / pi1 
        theta_hat = (RR2[:,t,:1] * Y - (RR1[:,t,:1] - 1) * mu1_1   - (RR2[:,t,:1] - RR1[:,t,:1]) * mu2_1)
        pred_theta[t,0] = torch.mean(theta_hat)
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
        f2[:,t,:1], f1[:,t,:1] = mu2_1, mu1_1
    elif d[0,0] == 0:
        RR2[:,t,:1], RR1[:,t,:1] = (1 - D1) * (1 - D2) / ((1 - pi1  ) * (1 - pi2_0  )), (1 - D1) / (1 - pi1)
        theta_hat = (RR2[:,t,:1] * Y - (RR1[:,t,:1] - 1) * mu1_0   - (RR2[:,t,:1] - RR1[:,t,:1]) * mu2_0)
        pred_theta[t,0] = torch.mean(theta_hat)
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
        f2[:,t,:1], f1[:,t,:1] = mu2_0, mu1_0


    #---------------------------------------------------------------------------
    # # Estimator 2: Bradic
    # try:
    #     bradic_result = estimateBradic(Y, X, D, X_index, folds)
    #     if d[0,0] == 1:
    #         pred_theta[t,1], pred_sig[t,1] = bradic_result[7], torch.sqrt(torch.tensor(bradic_result[10]))
    #     elif d[0,0] == 0:
    #         pred_theta[t,1], pred_sig[t,1] = bradic_result[13], torch.sqrt(torch.tensor(bradic_result[16]))
    # except:
    #     pred_theta[t,1], pred_sig[t,1] = 1000,1000
    #---------------------------------------------------------------------------


    # # Estimator 3: LASSO, RF, Net
    net_a_settings['estimate'] = False
    net_f_settings['estimate'] = False
    lasso_a_settings['c1'] = 0.1
    lasso_f_settings['c1'] = 0.1

    pred_theta[t,2:], pred_sig[t,2:], RR1[:,t,2:], RR2[:,t,2:], f1[:,t,2:], f2[:,t,2:] = estimateDynamicRiesz_all(Y, X, D, d, X_index, folds, 
                            lasso_a_settings = lasso_a_settings,
                                lasso_f_settings = lasso_f_settings,
                                        rf_a_settings = rf_a_settings,
                                            rf_f_settings = rf_f_settings,
                                                net_a_settings = net_a_settings,
                                                    net_f_settings = net_f_settings)

    if t % 10 == 0:
        print("Time until iteration ",t, ": ", time.time() - time0)

print("Finished. Total time: ", time.time() - time0)


Time until iteration  0 :  11.567918062210083
Time until iteration  10 :  127.63265991210938
Finished. Total time:  236.41983199119568


## Compute statistics

#### Get true value

In [11]:
true_value = theta1

#### Compute statistics

In [12]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method f_a": ["Oracle", "Bradic", "LASSO_LASSO", "LASSO_RF", "LASSO_Net", "RF_LASSO", "RF_RF", "RF_Net", "Net_LASSO", "Net_RF", "Net_Net"],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

#### Print results

In [13]:
df = pd.DataFrame(data)
print(df)

     Method f_a      Bias      RMSE  Coverage  Interval Length
0        Oracle -0.022764  0.360114      1.00         1.449431
1        Bradic -6.574519  6.574519      0.00         0.000000
2   LASSO_LASSO  0.686494  0.700992      0.05         0.763429
3      LASSO_RF  0.490554  0.572898      0.55         1.087189
4     LASSO_Net -6.574519  6.574519      0.00         0.000000
5      RF_LASSO -0.165801  0.387397      0.80         0.840631
6         RF_RF -0.035414  0.443048      0.80         1.242447
7        RF_Net -6.574519  6.574519      0.00         0.000000
8     Net_LASSO -6.574519  6.574519      0.00         0.000000
9        Net_RF -6.574519  6.574519      0.00         0.000000
10      Net_Net -6.574519  6.574519      0.00         0.000000


In [15]:
pred_theta[:,[0,2,3,4,5,6]]

tensor([[7.0758, 7.4819, 7.5333, 0.0000, 6.8087, 7.0382],
        [7.1137, 7.4902, 7.4802, 0.0000, 7.2213, 7.2522],
        [7.1051, 7.4426, 7.2328, 0.0000, 6.7511, 6.9038],
        [6.6081, 7.3609, 7.3116, 0.0000, 6.4124, 6.6567],
        [5.9134, 7.2576, 6.4485, 0.0000, 6.4700, 5.7709],
        [6.6796, 7.3097, 7.0003, 0.0000, 6.3251, 6.4798],
        [6.2497, 7.2275, 7.0700, 0.0000, 6.2361, 6.7052],
        [6.8552, 7.4428, 7.2593, 0.0000, 6.4013, 6.5872],
        [6.2062, 7.2890, 6.9694, 0.0000, 5.9012, 5.7918],
        [6.6767, 7.1379, 7.1453, 0.0000, 6.5907, 6.9695],
        [6.0272, 6.9381, 7.0592, 0.0000, 6.3125, 6.4877],
        [6.6142, 7.3315, 7.3099, 0.0000, 6.3108, 6.4512],
        [6.7217, 7.2202, 7.0966, 0.0000, 6.3766, 6.5095],
        [6.1090, 7.0133, 6.4614, 0.0000, 5.6424, 5.7947],
        [6.2819, 7.2460, 7.0333, 0.0000, 6.1970, 6.3232],
        [6.3246, 7.1343, 6.5247, 0.0000, 6.1538, 6.1124],
        [6.8079, 7.2887, 7.3158, 0.0000, 6.7208, 6.7850],
        [6.183

### RR MSE

In [14]:
torch.sqrt(torch.mean( torch.mean( (RR1[:,:,:1] - RR1[:,:,1:]) ** 2,0),0))

tensor([3.3659, 2.6236, 0.9516, 3.3659, 2.6236, 0.9516, 3.3659, 3.3659, 3.3659,
        3.3659])

### regresion MSE

In [18]:
torch.sqrt(torch.mean( torch.mean( (f2[:,:,:1] - f2[:,:,1:]) ** 2,0),0))

tensor([8.1165, 1.3022, 1.3022, 8.1165, 2.1889, 2.1889, 8.1165, 8.1165, 8.1165,
        8.1165])